# Tema: SCD Type 1

## Objetivos
Mantener una fila actual por cliente y rechazar cambios antiguos.

## Conceptos importantes para el examen
SCD1 sobrescribe atributos; no conserva versiones históricas de negocio. Time travel no sustituye el modelo de dimensión. Clave y secuencia determinan el estado.

**Dificultad:** Intermedio · **Tiempo estimado:** 60 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_20_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
customers = spark.createDataFrame(
    [(i, f"Cliente {i:02d}", ["Madrid", "Sevilla", "Bilbao"][i % 3],
      datetime(2026, 1, 1)) for i in range(1, 13)],
    "customer_id INT, name STRING, city STRING, updated_at TIMESTAMP"
)
customers.write.format("delta").mode("errorifexists").saveAsTable("customers")
updates = spark.createDataFrame([
    (2, "Cliente 02", "Valencia", datetime(2026, 2, 1)),
    (5, "Cliente 05", "Zaragoza", datetime(2026, 2, 2)),
    (13, "Cliente 13", "Madrid", datetime(2026, 2, 3))],
    customers.schema)
updates.write.format("delta").mode("errorifexists").saveAsTable("customers_updates")
display(customers)

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Dimensión actual

In [ ]:
%sql
CREATE TABLE dim_customer USING DELTA AS SELECT * FROM customers;
MERGE INTO dim_customer t USING customers_updates s ON t.customer_id=s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;

### 2. Leer estado resultante

In [ ]:
%sql
SELECT * FROM dim_customer ORDER BY customer_id;
SELECT customer_id, COUNT(*) n FROM dim_customer GROUP BY customer_id HAVING COUNT(*) > 1;

### 3. Reducir múltiples cambios por clave

In [ ]:
multi = updates.unionByName(updates.filter("customer_id=2").withColumn("city",F.lit("Lugo")).withColumn("updated_at", F.to_timestamp(F.lit("2026-03-01"))))
w = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())
latest = multi.withColumn("rn", F.row_number().over(w)).filter("rn=1").drop("rn")
latest.createOrReplaceTempView("latest_scd1")

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Construye dim_practice desde customers y aplica customers_updates.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Aplica latest_scd1 y comprueba ciudad de cliente 2 y unicidad de claves.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Reenvía customers_updates y demuestra que no revierte el cliente 2.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Crea una vista Gold con número de clientes por ciudad actual.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Simula dos eventos con mismo timestamp y distinto event_seq; elige el último de forma determinista.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** MERGE con condición temporal.

**Pista 2:** Solo gana la versión de marzo.

**Pista 3:** La condición temporal hace el replay seguro.

**Pista 4:** SCD1 responde al presente.

**Pista 5:** Ordena por fecha y secuencia, no por llegada.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
%sql
CREATE OR REPLACE TABLE dim_practice USING DELTA AS SELECT * FROM customers;
MERGE INTO dim_practice t USING customers_updates s ON t.customer_id=s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;

### Solución 2

In [ ]:
spark.sql("""MERGE INTO dim_practice t USING latest_scd1 s ON t.customer_id=s.customer_id
WHEN MATCHED AND s.updated_at>t.updated_at THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
assert spark.table("dim_practice").filter("customer_id=2").first().city == "Lugo"
assert spark.table("dim_practice").groupBy("customer_id").count().filter("count>1").count() == 0

### Solución 3

In [ ]:
spark.sql("""MERGE INTO dim_practice t USING customers_updates s ON t.customer_id=s.customer_id
WHEN MATCHED AND s.updated_at>t.updated_at THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
assert spark.table("dim_practice").filter("customer_id=2").first().city == "Lugo"

### Solución 4

In [ ]:
%sql
CREATE OR REPLACE VIEW customers_by_city AS SELECT city, COUNT(*) customers FROM dim_practice GROUP BY city;
SELECT * FROM customers_by_city;

### Solución 5

In [ ]:
ties = spark.createDataFrame([(2,"León",datetime(2026,4,1),10),(2,"Oviedo",datetime(2026,4,1),11)], "customer_id INT, city STRING, updated_at TIMESTAMP, event_seq LONG")
w = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc(),F.col("event_seq").desc())
winner = ties.withColumn("rn",F.row_number().over(w)).filter("rn=1")
assert winner.first().city == "Oviedo"
# En producción guarda también event_seq en el destino y compara ambas columnas en MERGE.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué conserva una dimensión SCD1?

A. Todas las versiones por clave

B. El estado actual

C. Solo las bajas

D. Un checkpoint

### Pregunta 2
¿Qué evita que un evento antiguo deshaga uno nuevo?

A. ORDER BY al consultar

B. OPTIMIZE

C. Comparar secuencia al aplicar cambios

D. Cambiar el nombre de la tabla

### Pregunta 3
Necesitas saber la ciudad del cliente en la fecha de una compra. ¿Qué modelo suele encajar?

A. SCD2

B. SCD1 sin historial

C. Vista temporal

D. COPY INTO por sí solo

### Respuestas y explicación
**1. B** — Los atributos anteriores son sobrescritos.

**2. C** — La protección debe estar en la escritura.

**3. A** — SCD2 representa periodos de validez.

## PARTE 6 - RETO FINAL
Añade deleted como baja lógica y una secuencia de origen. Aplica eventos fuera de orden sin resucitar clientes eliminados por un evento posterior.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
